WARNING: Notebook was written during development. May not work as intended. Retained here only for completion and future reference.

# Swelling of an annular gel-elastomer rod

Forward analysis only, for the method figure: an annular rod whose wall is tiled by
rings of angular sectors, each sector either hydrogel or elastomer. A small library of
closed-form layouts places the gel as a function of arc length `s`, radial position `r`
and polar angle `θ`, the rod is swollen with the Flory-Rehner model, and the reference
layout is shown beside the swollen shape.

In [ ]:
import pathlib

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Wedge

import crops.src.bc as _bc
import crops.src.fe_cosserat as _fe
import crops.src.mesher as _mesher
import crops.src.solver as _solver
import crops.src.utils as _utils
import crops.src.viz as _viz

jax.config.update("jax_enable_x64", True)
plt.rcParams.update(_viz.default_plot_settings)


figure_dir = pathlib.Path("crops/notebooks/dev/paper/figures")
figure_dir.mkdir(parents=True, exist_ok=True)

### Annular cross-section

The wall between `inner_radius` and `outer_radius` is split into `num_rings` rings of
`num_sectors` sectors each. Every sector is one fiber of the rod model: its centroid is
the quadrature point and its area the quadrature weight, so the resolution of the
material layout in the cross-section is set here.

In [ ]:
rod_length = 1.0
num_elems = 120
num_nodes = num_elems + 1
parametric_coords = jnp.linspace(0.0, rod_length, num_nodes)

outer_radius = rod_length / 30.0
inner_radius = 0.5 * outer_radius
num_rings = 4
num_sectors = 16
num_fibers = num_rings * num_sectors

# Sector layout, ring-major from the inner wall outwards, counter-clockwise from d1.
ring_edges = np.linspace(inner_radius, outer_radius, num_rings + 1)
sector_edges = np.linspace(0.0, 2.0 * np.pi, num_sectors + 1)
ring_index, sector_index = np.divmod(np.arange(num_fibers), num_sectors)
sector_inner = ring_edges[ring_index]
sector_outer = ring_edges[ring_index + 1]
sector_start = sector_edges[sector_index]
sector_end = sector_edges[sector_index + 1]

# Closed-form area, centroid and centroidal second moments of an annular sector.
span = sector_end - sector_start
sector_areas = 0.5 * span * (sector_outer**2 - sector_inner**2)
cubic = (sector_outer**3 - sector_inner**3) / 3.0
centroid_d1 = cubic * (np.sin(sector_end) - np.sin(sector_start)) / sector_areas
centroid_d2 = cubic * (np.cos(sector_start) - np.cos(sector_end)) / sector_areas
quartic = (sector_outer**4 - sector_inner**4) / 4.0
double_angle = 0.25 * (np.sin(2.0 * sector_end) - np.sin(2.0 * sector_start))
moi_about_d1 = quartic * (0.5 * span - double_angle) - sector_areas * centroid_d2**2
moi_about_d2 = quartic * (0.5 * span + double_angle) - sector_areas * centroid_d1**2


def along_rod(values):
  """Repeat a per-fiber quantity over every element."""
  return np.broadcast_to(values, (num_elems,) + np.shape(values))


rod = _mesher.build_rod_geometry(
  analytical_curve=lambda s: jnp.array([s, 0.0, 0.0]),
  parametric_coords=parametric_coords,
  fiber_offsets=jnp.asarray(along_rod(np.stack([centroid_d1, centroid_d2], axis=-1))),
  fiber_areas=jnp.asarray(along_rod(sector_areas)),
  fiber_mass_density=jnp.full((num_elems, num_fibers), 1100.0),
  fiber_centroid_moi=jnp.asarray(along_rod(np.stack([moi_about_d1, moi_about_d2], -1))),
  seed_binormal=jnp.array([0.0, 0.0, 1.0]),
)

# Sector geometry in the layout `_viz.plot_rod_annular_sections` expects.
section = dict(
  fiber_inner_radii=along_rod(sector_inner),
  fiber_outer_radii=along_rod(sector_outer),
  fiber_start_angles=along_rod(sector_start),
  fiber_end_angles=along_rod(sector_end),
)

# Normalised coordinates of every element-fiber cell for the layout functions:
# s in (0, 1) along the rod, r in (0, 1) across the wall, theta in [0, 2pi) from d1.
s_hat = np.asarray(0.5 * (parametric_coords[:-1] + parametric_coords[1:])) / rod_length
r_hat = (np.hypot(centroid_d1, centroid_d2) - inner_radius) / (
  outer_radius - inner_radius
)
theta = np.arctan2(centroid_d2, centroid_d1) % (2.0 * np.pi)
S, R, TH = np.broadcast_arrays(s_hat[:, None], r_hat[None, :], theta[None, :])

print(f"{num_fibers} sectors per section, {num_elems * num_fibers} cells")
print(f"wall / outer radius = {(outer_radius - inner_radius) / outer_radius:.2f}")
print(f"length / outer diameter = {rod_length / (2 * outer_radius):.0f}")

### Material layouts

Each layout returns a signed *signal* on the `(num_elems, num_fibers)` cells: positive is
gel, negative is elastomer. `project` turns that into a gel fraction, either as a hard
two-phase step or as a graded `tanh` of chosen sharpness. Gel swells and elastomer does
not, so the rod bends away from the gel; the shape follows from how the gel side moves
around the section along the length, and how deep into the wall it reaches.

Layouts compose with plain numpy: `np.minimum(a, b)` is gel where both are gel,
`np.maximum(a, b)` where either is, and `-a` swaps the phases.

In [ ]:
def project(signal, sharpness=None):
  """Gel fraction in [0, 1] from a signed signal. Positive is gel, negative elastomer.

  With `sharpness=None` the split is a hard two-phase step; otherwise a tanh of that
  steepness gives a graded transition.
  """
  if sharpness is None:
    return (signal > 0.0).astype(float)
  return 0.5 * (1.0 + np.tanh(sharpness * signal))


def bilayer(s, r, theta, split_angle=0.0):
  """Gel on the half of the annulus facing `split_angle`: a bilayer rolled into a tube.

  Bends in one plane with uniform curvature, so the rod becomes a circular arc.
  """
  return np.cos(theta - split_angle)


def helical_stripe(s, r, theta, num_turns=0.5, half_width=np.pi / 2):
  """A gel stripe of angular half-width `half_width` winding `num_turns` times around
  the rod. The bending direction rotates with `s`, so the rod coils into a helix whose
  torsion is set by `num_turns`; many turns at fixed curvature gives a gentle wave
  rather than a coil, so keep `2 pi num_turns` below the bilayer's `kappa L`.
  """
  return np.cos(theta - 2.0 * np.pi * num_turns * s) - np.cos(half_width)


def rotating_segments(s, r, theta, num_segments=3, angle_step=np.pi):
  """Bilayer whose gel side jumps by `angle_step` at every one of `num_segments`
  equal segments. `pi` gives a planar serpentine, `pi / 2` a three-dimensional zigzag.
  """
  segment = np.floor(num_segments * s)
  return np.cos(theta - angle_step * segment)


def wobbling_bilayer(s, r, theta, num_waves=2.0, amplitude=np.pi / 3):
  """Bilayer whose gel side swings sinusoidally about `d1` by up to `amplitude`
  `num_waves` times along the rod, so the bending plane wanders out of plane.
  """
  return np.cos(theta - amplitude * np.sin(2.0 * np.pi * num_waves * s))


def thickening_bilayer(s, r, theta, split_angle=0.0, power=0.5):
  """Bilayer whose gel half deepens from the outer wall inwards along the rod, being
  the rings with `r > 1 - s ** power`. The root is all elastomer and the tip has gel
  through the full wall, so the curvature grows along the length and the rod spirals.
  """
  return np.minimum(np.cos(theta - split_angle), r - (1.0 - s**power))


# Candidate layouts. Add or edit entries here, then pick one below.
designs = {
  "arc": lambda: bilayer(S, R, TH),
  "helix": lambda: helical_stripe(S, R, TH, num_turns=0.5, half_width=np.pi / 2),
  "serpentine": lambda: rotating_segments(S, R, TH, num_segments=3, angle_step=np.pi),
  "zigzag": lambda: rotating_segments(S, R, TH, num_segments=3, angle_step=np.pi / 2),
  "wobble": lambda: wobbling_bilayer(S, R, TH, num_waves=2.0, amplitude=np.pi / 3),
  "spiral": lambda: thickening_bilayer(S, R, TH, power=0.5),
  # Outer-wall-only helix: gel where the helical stripe AND the outer half of the wall.
  "shell_helix": lambda: np.minimum(helical_stripe(S, R, TH, num_turns=0.5), R - 0.5),
}
sharpness = None  # None for two-phase, e.g. 6.0 for a graded layout

### Cross-section preview

End-on sections at a few stations, drawn as wedges in the `d1`-`d2` plane, to check a
layout before solving.

In [ ]:
background_color = "#C8F5FF"
plt.rcParams.update(
  {
    "figure.facecolor": background_color,
    "axes.facecolor": background_color,
    "savefig.facecolor": background_color,
  }
)

In [ ]:
def draw_section(ax, gel_fraction_at_elem, title=None):
  """Draw one cross-section as coloured annular-sector wedges in the d1-d2 plane."""
  for f in range(num_fibers):
    ax.add_patch(
      Wedge(
        center=(0.0, 0.0),
        r=sector_outer[f],
        theta1=np.degrees(sector_start[f]),
        theta2=np.degrees(sector_end[f]),
        width=sector_outer[f] - sector_inner[f],
        facecolor=_viz.elast_gel_cmap(gel_fraction_at_elem[f]),
        edgecolor="k",
        linewidth=0.2,
      )
    )
  ax.set_xlim(-1.05 * outer_radius, 1.05 * outer_radius)
  ax.set_ylim(-1.05 * outer_radius, 1.05 * outer_radius)
  ax.set_aspect("equal")
  ax.set_axis_off()
  if title is not None:
    ax.set_title(title, fontsize=9)


def preview_sections(gel_fraction, stations=(0.05, 0.3, 0.5, 0.7, 0.95), title=None):
  """A row of cross-sections at the given fractions of the rod length."""
  fig, axes = plt.subplots(1, len(stations), figsize=(2.2 * len(stations), 2.4))
  for ax, station in zip(axes, stations):
    elem = min(int(station * num_elems), num_elems - 1)
    draw_section(ax, gel_fraction[elem], title=f"s/L = {station:.2f}")
  if title is not None:
    fig.suptitle(title, fontsize=10)
  fig.tight_layout()
  return fig


for name, layout in designs.items():
  preview_sections(project(layout(), sharpness), title=name)
plt.show()

### Material, environment and boundary conditions

The gel fraction interpolates the shear modulus and the Flory parameter between the two
phases. The bath chemical potential is ramped from dry to wet over the load steps. The
root is clamped and the rest of the rod is free to move in three dimensions.

In [ ]:
# The stiffness contrast sets how much the elastomer restrains the gel. For the plain
# bilayer at this aspect ratio: 100x barely bends (kappa L ~ 1), 10x turns ~3.9 rad,
# 5x turns ~4.6 rad, 3x closes a loop (~5 rad).
shear_modulus_gel, shear_modulus_elastomer = 1.0e6, 5.0e6
chi_gel, chi_elastomer = 0.3, 2.5

solvent = _fe.FlorySolvent(
  bath_temperature=298.0,
  molar_volume=1.8e-5,
  ref_chem_potential=0.0,
)

num_steps = 20
ramp = jnp.linspace(0.0, 1.0, num_steps + 1)[1:] ** 0.03
dry_chem_potential = -1.0e5
wet_chem_potential = -350.0  # less negative swells more
chem_potentials = dry_chem_potential + (wet_chem_potential - dry_chem_potential) * ramp

clamped_dofs = (
  _utils.Translation.X,
  _utils.Translation.Y,
  _utils.Translation.Z,
  _utils.Rotation.THETA_X,
  _utils.Rotation.THETA_Y,
  _utils.Rotation.THETA_Z,
)
boundary_list = [
  _bc.DirichletBC(
    values=(dof, [np.array([0.0]) for _ in range(num_steps)]),
    node_targets=[np.array([0]) for _ in range(num_steps)],
    name=f"clamp_{dof}",
  )
  for dof in clamped_dofs
]
bc = _bc.process_boundary_conditions(rod, boundary_list)

nr_solver = _solver.NewtonRaphson(
  solver_type=_solver.NonlinearSolvers.ARMIJO_BACKTRACKING_NEWTON_RAPHSON,
  nonlinear_settings={
    "max_iter": 100,
    "threshold": 1e-6,
    "abs_threshold": 1e-6,
    "lam_min": 0.01,
    "line_search_max_iter": 12,
    "line_search_shrink": 0.5,
    "line_search_alpha_min": 1e-6,
    "line_search_armijo_c": 1e-4,
  },
  linear_settings={
    "solver": _solver.LinearSolvers.SCIPY_SPARSE,
    "rtol": 1e-8,
  },
)
problem = _fe.CosseratProblem(
  rod=rod,
  bc=bc,
  nr_solver=nr_solver,
  solvent=solvent,
  bisection_settings={
    "lower": 1e-6,
    "upper": 1.0 - 1e-6,
    "tol": 1e-6,
    "max_iter": 200,
  },
)


def solve_layout(gel_fraction):
  """Swell the rod for a gel fraction field and return the deformed centreline
  coordinates and directors."""
  gel_fraction = jnp.asarray(gel_fraction)
  u_star, _ = problem.solve(
    u0=jnp.zeros(rod.num_dofs),
    shear_moduli=(
      shear_modulus_gel * gel_fraction + shear_modulus_elastomer * (1.0 - gel_fraction)
    ),
    flory_parameters=chi_gel * gel_fraction + chi_elastomer * (1.0 - gel_fraction),
    chem_potentials=chem_potentials,
  )
  coords, directors, _ = _mesher.extract_coords_and_director_from_deformation(
    rod, u_star
  )
  return np.asarray(coords), np.asarray(directors)

### Gallery

Swell every candidate layout and show the reference rod beside the swollen one. The
cross-section is drawn with its reference radii in both states; the model does not
track the swollen wall thickness.

In [ ]:
def draw_rod(
  ax, coords, directors, gel_fraction, view=(25.0, -60.0), num_arc_segments=1
):
  """Render the annular rod coloured by gel fraction on a 3D axis.

  The viewport is fitted to the rod's own bounding box at equal aspect, with slender
  directions held open to a fraction of the longest one, rather than to a cube.
  """
  _viz.plot_rod_annular_sections(
    node_coords=coords,
    directors=directors,
    **section,
    num_arc_segments=num_arc_segments,
    fiber_field=gel_fraction,
    fiber_field_val_range=(0.0, 1.0),
    cmap=_viz.elast_gel_cmap,
    edge_width=0.1,
    ax=ax,
  )
  low = coords.min(axis=0) - 1.5 * outer_radius
  high = coords.max(axis=0) + 1.5 * outer_radius
  span = np.maximum(high - low, 0.25 * np.max(high - low))
  mid = 0.5 * (low + high)
  ax.set_xlim3d(mid[0] - 0.5 * span[0], mid[0] + 0.5 * span[0])
  ax.set_ylim3d(mid[1] - 0.5 * span[1], mid[1] + 0.5 * span[1])
  ax.set_zlim3d(mid[2] - 0.5 * span[2], mid[2] + 0.5 * span[2])
  ax.set_box_aspect(span)
  ax.view_init(elev=view[0], azim=view[1])


In [ ]:


results = {}
for name, layout in designs.items():
  gel_fraction = project(layout(), sharpness)
  coords, directors = solve_layout(gel_fraction)
  results[name] = (gel_fraction, coords, directors)

  kinematics = _mesher.compute_discrete_cosserat_kinematics(
    node_coords=jnp.asarray(coords),
    directors=jnp.asarray(directors),
    arc_coords=rod.arc_coords,
  )
  max_curvature = np.abs(np.asarray(kinematics.curvature)).max(axis=0) * rod_length
  chords = np.linalg.norm(np.diff(coords, axis=0), axis=1)
  print(
    f"{name:12s} arc length / L = {chords.sum() / rod_length:.3f}   "
    f"max |kappa| L = {np.array2string(max_curvature, precision=2)}"
  )

  fig = plt.figure(figsize=(11, 4))
  ax_ref = fig.add_subplot(1, 2, 1, projection="3d")
  draw_rod(
    ax_ref, np.asarray(rod.node_coords), np.asarray(rod.directors_0), gel_fraction
  )
  ax_ref.set_title(f"{name}: reference", fontsize=9)
  ax_def = fig.add_subplot(1, 2, 2, projection="3d")
  draw_rod(ax_def, coords, directors, gel_fraction)
  ax_def.set_title(f"{name}: swollen", fontsize=9)
  fig.tight_layout()
  plt.show()

### Method figure

Pick a layout and render it for the paper: reference layout on the left, swollen shape
on the right, plus the cross-sections. Arc segments are refined here for smoother
rings; the file is saved as PNG because the polygon count makes a vector file heavy.

In [ ]:
design_name = "helix"
view = (25.0, -60.0)  # (elevation, azimuth); use (90, -90) for the x-y plane

gel_fraction, coords, directors = results[design_name]

fig = plt.figure(figsize=(12, 5))
# ax_ref = fig.add_subplot(1, 2, 1, projection="3d")
# draw_rod(
#   ax_ref,
#   np.asarray(rod.node_coords),
#   np.asarray(rod.directors_0),
#   gel_fraction,
#   view=view,
#   num_arc_segments=2,
# )
ax_def = fig.add_subplot(1, 1, 1, projection="3d")
draw_rod(ax_def, coords, directors, gel_fraction, view=view, num_arc_segments=2)
fig.tight_layout()
fig.savefig(figure_dir / f"annular_swell_{design_name}.png", dpi=300)
plt.show()